In [2]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

In [3]:

@dataclass
class Message:
    content: str

In [4]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

In [5]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [6]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

In [7]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [8]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [9]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [10]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are several reasons in favor of choosing AutoGen for your AI Agent project:

1. **Collaborative Capabilities**: AutoGen enables agents to engage in conversations, fostering cooperative interactions that can enhance problem-solving and idea generation (divergent thinking).

2. **Improved Factuality and Reasoning**: The architecture supports improved accuracy in the information provided by AI agents, as well as enhanced logical reasoning capabilities.

3. **Structured Approach**: It provides guardrails that help manage the behavior of the agents, ensuring safe and responsible use while performing tasks.

4. **Task Decomposition**: AutoGen assists in breaking down complex tasks into simpler sub-tasks, making it easier to manage and structure sophisticated applications.

5. **Versatility Across Domains**: The framework is adaptable across various fields, allowing for the development of advanced applications tailored to different user needs and use cases.

These advantages make AutoGen a compelling choice for developing robust AI agents that require collaboration, accuracy, and task management. 

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen in an AI agent project:

1. **Unpredictable Workflows**: The flexibility offered by AutoGen can sometimes result in workflows that are not well-defined, leading to unpredictability in how tasks are executed.

2. **Risk of Self-Sabotage**: Agents might engage in loops or ramble, causing inefficiencies or errors as they could inadvertently fall into repetitive patterns or fail to make progress.

3. **Complexity in Management**: The communication between multiple AI agents can create complexities that make it harder to monitor and manage interactions effectively.

4. **Resource Intensive**: The need for potentially continuous oversight and troubleshooting can require increased resources, including computational power and human intervention.

5. **Integration Challenges**: Integrating AutoGen with existing systems may present challenges, especially if those systems are not designed to accommodate the flexibility and freeform nature of the framework.

6. **Learning Curve**: The architecture may have a steep learning curve for developers who need to understand how to effectively manage and utilize the AutoGen capabilities.

These points could influence the decision not to use AutoGen in your project. 

TERMINATE



## Decision:

Based on the research provided by the team, I recommend using AutoGen for the AI Agent project. The pros clearly demonstrate significant benefits, particularly in collaboration, improved reasoning, and task management, which are crucial for developing effective AI agents. While the cons present valid concerns regarding unpredictability and resource intensity, these can often be mitigated through careful planning and management strategies. Ultimately, the advantages of enhanced collaboration, accuracy, and versatility outweigh the potential drawbacks, making AutoGen a strong candidate for this project. 

TERMINATE

In [11]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [12]:
await host.stop()